In [ ]:
# ROB-KN01 (spec v1.2 "Robustness embedder: nomic-embed-text-v1.5, design before any code", 2026-11-19): nomic vectors for
# the 34 setups (17 settings x 2 parsers) and their questions, then scorer_v1 with hybrid = BM25 + nomic (RRF k = 60).
# Tuning documents only (dataset ROB-KD01 from scripts/rob_kaggle_bundle.py); no --final anywhere.
# Needs: GPU T4, Internet on (clone + model download), dataset ROB-KD01 attached. No secrets.
COMMIT = ""  # the commit that holds the nomic code; set before running
assert COMMIT, "set COMMIT to the commit with the robustness embedder code"
!git clone -q https://github.com/amanchaurasia512/Self_optimized_Agent_Ai_chunking.git /kaggle/working/repo
!cd /kaggle/working/repo && git checkout -q {COMMIT} && git log --oneline -1 && git status --porcelain | head

In [ ]:
# The check cell imports rob_analyse -> p4_analyse -> src.chunk.chunkers, which needs stanza and langchain at import
# time. Pins from requirements.txt (as in the ROB-KN01 Kaggle run).
!pip install -q stanza==1.15.0 langchain-text-splitters==1.1.3 langchain-community==0.4.2 langchain-experimental==0.4.2

In [ ]:
# Same pins as the study .venv and P3-KN04. nomic loads through the native transformers nomic_bert class
# (trust_remote_code off: the remote code fails on transformers 5.22), so no einops is needed.
!pip install -q sentence-transformers==6.2.0 transformers==5.22.1
!pip freeze > /kaggle/working/rob-kn01_pip_freeze.txt
!grep -i -E "^(torch|transformers|sentence-transformers|numpy|pandas)==" /kaggle/working/rob-kn01_pip_freeze.txt
import torch
print("GPU:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
assert torch.cuda.is_available(), "turn the GPU on"

In [ ]:
# Copy ROB-KD01 into the clone at its repo paths. Every file must be listed in BUNDLE.json with the same SHA-256 and
# belong to a tuning document; anything else stops the notebook before a run.
import hashlib
import json
import shutil
import sys
import zipfile
from pathlib import Path

REPO, INPUT, TEMP = Path("/kaggle/working/repo"), Path("/kaggle/input"), Path("/kaggle/temp/rob-kd01")
sys.path.insert(0, str(REPO))
from scripts.rob_analyse import robustness_settings
from src.common.manifest import code_version
from src.common.split import tuning_ids

record_path = next(INPUT.rglob("BUNDLE.json"))
bundle = record_path.parent
if not (bundle / "data").is_dir():  # uploaded with --dir-mode zip and not unpacked by Kaggle
    for archive in bundle.glob("*.zip"):
        zipfile.ZipFile(archive).extractall(TEMP / archive.stem)
    bundle = TEMP
record, tuning, settings = json.loads(record_path.read_text()), tuning_ids(), robustness_settings()
assert set(record["documents"]) == tuning, "ROB-KD01 documents differ from the tuning split of this commit"
assert record["settings"] == settings and record["setups"] == 34, "ROB-KD01 settings differ from the 34 setups of this commit"
sha = lambda path: hashlib.sha256(path.read_bytes()).hexdigest()
for entry in record["files"]:
    source, target = bundle / entry["path"], REPO / entry["path"]
    assert Path(entry["path"]).stem in tuning and sha(source) == entry["sha256"], entry["path"]
    target.parent.mkdir(parents=True, exist_ok=True)
    shutil.copyfile(source, target)
for path in (bundle / "data" / "gold").glob("*.jsonl"):
    target = REPO / path.relative_to(bundle)
    target.parent.mkdir(parents=True, exist_ok=True)
    shutil.copyfile(path, target)
gold_docs = {json.loads(line)["doc_id"] for path in (REPO / "data" / "gold").glob("*.jsonl") for line in path.open()}
assert gold_docs <= tuning, sorted(gold_docs - tuning)
assert not (REPO / "data" / "embeddings" / "nomic-v1.5").exists(), "the nomic cache must start empty"
for folder in ["chunks/docling", "chunks/pdfplumber", "extracted/docling", "extracted/pdfplumber", "gold"]:
    print(folder, ":", sum(path.is_file() for path in (REPO / "data" / folder).rglob("*")), "files")
print("tuning documents:", len(tuning), "| settings per parser:", len(settings), "| code:", code_version())

In [ ]:
# Step 1: nomic vectors (GPU, float32, pinned revision, 768 dimensions, normalised) for every chunk text of the 34
# setups and every tuning question; "search_document: " / "search_query: " are added when encoding only.
SETTINGS = " ".join(settings)
!cd /kaggle/working/repo && python -m scripts.p3_embed --embedder nomic --settings {SETTINGS} --name ROB-KN01-full --machine kaggle --run-source ROB-KN01

In [ ]:
# Step 2: scorer_v1 on the 34 setups with the nomic cache for the dense part, as a new run P3-ROB-KN01-...
!cd /kaggle/working/repo && python -m scripts.p3_score --embedder nomic --settings {SETTINGS} --machine kaggle --run-source ROB-KN01

In [ ]:
# Step 3: one zip of every output and run record, at repo paths (unzip at the repo root on the laptop).
files = sorted((REPO / "data" / "embeddings" / "nomic-v1.5").iterdir())
files += sorted((REPO / "data" / "results" / "p3").glob("P3-ROB-KN01-*")) + sorted((REPO / "runs").glob("P3-ROB-KN01-*.json"))
with zipfile.ZipFile("/kaggle/working/rob-kn01_outputs.zip", "w", zipfile.ZIP_DEFLATED) as archive:
    for path in files:
        archive.write(path, path.relative_to(REPO))
    archive.write("/kaggle/working/rob-kn01_pip_freeze.txt", "data/external/kaggle_out/ROB-KN01/rob-kn01_pip_freeze.txt")
print(len(files), "files zipped | run records:", sorted(path.name for path in files if path.parent.name == "runs"))
!ls -la /kaggle/working